In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/HHS_Unaccompanied_Alien_Children_Program.csv")
df.columns = ["date", "apprehended", "cbp_custody",
              "cbp_transferred", "hhs_care", "hhs_discharged"]
df = df.dropna(how="all").copy()
df["hhs_care"] = df["hhs_care"].str.replace(",", "").astype(int)

print(df.shape)         # (720, 6)
print(df["date"].iloc[0], type(df["date"].iloc[0]))

(720, 6)
December 21, 2025 <class 'str'>


In [2]:
df["date"] = pd.to_datetime(df["date"], format="%B %d, %Y")
df["date"].dtype      # datetime64

dtype('<M8[ns]')

In [3]:
df = df.sort_values("date").reset_index(drop=True)
print(df["date"].min(), df["date"].max())
print(df["date"].is_monotonic_increasing)   # True

2023-01-12 00:00:00 2025-12-21 00:00:00
True


In [4]:
print("duplicate dates:", df["date"].duplicated().sum())
print("duplicate rows: ", df.duplicated().sum())

duplicate dates: 0
duplicate rows:  0


In [5]:
full_range = pd.date_range(df["date"].min(), df["date"].max(), freq="D")
print(len(full_range))     # 1075

daily = df.set_index("date").reindex(full_range)
daily.index.name = "date"
daily["is_reported"] = daily["hhs_care"].notna()

print(daily["is_reported"].value_counts())

1075
is_reported
True     720
False    355
Name: count, dtype: int64


In [6]:
missing = daily[~daily["is_reported"]]

# Which weekdays are missing?
print(missing.index.day_name().value_counts())

# How long are the gap runs?
run_id = (daily["is_reported"] != daily["is_reported"].shift()).cumsum()
run_lengths = missing.groupby(run_id[~daily["is_reported"]]).size()
print(run_lengths.value_counts().sort_index())

date
Saturday     154
Friday       152
Sunday        24
Monday         8
Thursday       7
Wednesday      6
Tuesday        4
Name: count, dtype: int64
1     10
2    122
3     23
4      3
5      1
6      1
9      1
Name: count, dtype: int64
